# Build the first modeling table (auto-picks best-covered drug)

In [1]:
from pathlib import Path
import os

# set project root = parent of /notebooks
PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("CWD set to:", Path.cwd())

CWD set to: c:\Users\josha\Codeacademy Projects\Data Scientist Portfolio Project


load processed files

In [2]:
from pathlib import Path
import pandas as pd

OUT = Path("data/processed")

expr = pd.read_parquet(OUT / "expression_tpm_logp1.parquet")
pr = pd.read_parquet(OUT / "prism_secondary_collapsed.parquet")

expr.shape, pr.shape

((1517, 19194), (644798, 8))

find the best-covered compound

In [3]:
coverage = (
    pr.groupby(["broad_id", "compound_name"])["depmap_id"]
      .nunique()
      .reset_index(name="n_cell_lines")
      .sort_values("n_cell_lines", ascending=False)
)

coverage.head(20)

,broad_id,compound_name,n_cell_lines
399,BRD-K13662825-001-07-5,dinaciclib,480
642,BRD-K32744045-001-31-2,disulfiram,480
1435,BRD-K95142244-001-01-5,talazoparib,480
718,BRD-K38852836-001-02-1,ganetespib,479
60,BRD-A25608658-001-01-1,narasin,479
504,BRD-K21361524-001-01-1,selinexor,479
1384,BRD-K89714990-001-01-5,tepoxalin,479
650,BRD-K33622447-066-01-9,abemaciclib,479
646,BRD-K33379087-001-07-5,tivantinib,479
633,BRD-K31928526-001-02-1,barasertib,479


merge to make modeling table (AUC)

In [4]:
top = coverage.iloc[0]
broad_id = top["broad_id"]
compound = top["compound_name"]
target_col = "auc"

print("Selected:", compound, "|", broad_id, "| n_cell_lines:", int(top["n_cell_lines"]))

pr_drug = pr.loc[pr["broad_id"] == broad_id, ["depmap_id", target_col]].dropna()
df_model = pr_drug.merge(expr, on="depmap_id", how="inner")

print("Modeling table shape:", df_model.shape)
df_model.head()

Selected: dinaciclib | BRD-K13662825-001-07-5 | n_cell_lines: 480
Modeling table shape: (476, 19195)


,depmap_id,auc,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,...,H3C2,H3C3,AC098582.1,DUS4L-BCAP29,C8orf44-SGK3,ELOA3B,NPBWR1,ELOA3D,ELOA3,CDR1
0,ACH-000007,0.677223,3.350497,0.000000,5.957218,3.129283,3.716991,0.028569,0.014355,5.630231,...,1.220330,0.000000,0.163499,2.182692,0.111031,0.000000,0.000000,0.000000,0.000000,0.0
1,ACH-000008,0.633223,2.887525,0.000000,7.188638,2.198494,4.116864,0.000000,0.097611,6.474274,...,0.454176,0.963474,0.847997,3.161888,0.176323,0.000000,0.903038,0.000000,0.000000,0.0
2,ACH-000011,0.644529,3.925999,0.000000,5.638074,2.117695,2.584963,0.000000,0.378512,5.575312,...,1.021480,0.084064,0.042644,1.157044,0.378512,0.000000,0.000000,0.000000,0.000000,0.0
3,ACH-000012,0.503696,5.158660,0.000000,6.132577,1.974529,3.779260,0.028569,2.746313,6.275380,...,0.739848,0.613532,1.028569,1.650765,0.189034,0.014355,0.163499,0.014355,0.014355,0.0
4,ACH-000013,0.714837,4.917432,0.056584,7.639232,1.914565,3.566815,0.028569,2.056584,5.455163,...,0.443607,0.000000,0.000000,1.432959,0.238787,0.028569,0.028569,0.000000,0.028569,0.0


save modeling table

In [5]:
safe_name = "".join(ch if ch.isalnum() else "_" for ch in compound.lower()).strip("_")
out_model = OUT / f"modeling_{safe_name}_{target_col}.parquet"
df_model.to_parquet(out_model, index=False)

print("Wrote:", out_model)

Wrote: data\processed\modeling_dinaciclib_auc.parquet
